<a href="https://colab.research.google.com/github/ShivaBandereddy/incident-triage-agent/blob/main/day3_hitl_rag.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import userdata
import os
os.environ['GOOGLE_API_KEY'] = userdata.get('GOOGLE_API_KEY')

In [2]:
!pip install -q langgraph langchain-google-genai chromadb sentence-transformers

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.62.2 which is incompatible.
holidays 0.108 requires python-dateutil<3,>=2.9.0.post0, but you have python-dateutil 4.0.0 which is incompatible.
google-adk 2.7.1 requires google-genai<3,>=2.12.1, but you have google-genai 3.3.0 which is incompatible.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.2 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.45.2 which is incompatible.
ibis-framework 9.5.0 requires python-dateutil<3,>=2.8.2, but you have python-dateutil 4.0.0 which is incompatible.
google-cloud-bigquery 3.47.0 requires python-dateutil<3.0.0,>=2.8.2, but you have python-dateutil 4.0.0 which is incompatible.
betterproto 2.0.

In [3]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver
from langgraph.types import interrupt, Command

class S(TypedDict):
    proposal: str
    approved: bool

def propose(state: S):
    return {'proposal': 'Move VM app-12 to ds-prod-02', 'approved': False}

def approval(state: S):
    decision = interrupt({'proposal': state['proposal'], 'question': 'Approve?'})
    return {'approved': decision['approved']}

g = StateGraph(S)
g.add_node('propose', propose)
g.add_node('approval', approval)
g.add_edge(START, 'propose')
g.add_edge('propose', 'approval')
g.add_edge('approval', END)
app = g.compile(checkpointer=MemorySaver())

cfg = {'configurable': {'thread_id': 't1'}}
out = app.invoke({'proposal': '', 'approved': False}, cfg)
print(out['__interrupt__'])

[Interrupt(value={'proposal': 'Move VM app-12 to ds-prod-02', 'question': 'Approve?'}, id='85b74ef059ecdb4fa5dbc2cee23d7a07', response_schema=None)]


In [4]:
print(app.invoke(Command(resume={'approved': True}), cfg))

{'proposal': 'Move VM app-12 to ds-prod-02', 'approved': True}


In [5]:
cfg2 = {'configurable': {'thread_id': 't2'}}
app.invoke({'proposal': '', 'approved': False}, cfg2)
print(app.invoke(Command(resume={'approved': False}), cfg2))

{'proposal': 'Move VM app-12 to ds-prod-02', 'approved': False}


In [6]:
import chromadb
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2')
client = chromadb.Client()  # in-memory, files avasaram ledu
col = client.get_or_create_collection('demo')

docs = [
    'Disk full: free space on datastore',
    'DNS failure: cannot resolve hostname',
    'Certificate expired: TLS handshake fails',
]
col.add(ids=['1', '2', '3'], documents=docs,
        embeddings=model.encode(docs).tolist())

q = 'TLS errors and handshake failed'
res = col.query(query_embeddings=model.encode([q]).tolist(), n_results=2,
                include=['documents', 'distances'])
print(res['documents'])
print(res['distances'])

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (4 outputs left out)

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

[['Certificate expired: TLS handshake fails', 'DNS failure: cannot resolve hostname']]
[[0.363476037979126, 1.2727233171463013]]


In [7]:
q2 = 'users report slowness on the application'
res2 = col.query(query_embeddings=model.encode([q2]).tolist(), n_results=3,
                 include=['documents', 'distances'])
print(res2['documents'])
print(res2['distances'])

[['Disk full: free space on datastore', 'DNS failure: cannot resolve hostname', 'Certificate expired: TLS handshake fails']]
[[1.750255823135376, 1.9577828645706177, 2.0393145084381104]]
